# DeepEF - Fast ProtT5 Embedding Generation (Colab GPU)

Downloads CASP12 protein sequences from SidechainNet, generates all 8 ProtT5 embedding variants on a T4 GPU,
and saves consolidated output files. No per-protein disk writes — everything stays in memory until final save.

**Steps:**
1. Set runtime to T4 GPU
2. Run all cells (~30-40 min total)
3. Download `casp12_data_30_with_embeddings.tar.gz` from Google Drive

In [ ]:
!pip install -q transformers sentencepiece sidechainnet openmm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 24.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 81.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 26.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 12.8 MB/s eta 0:00:00


In [ ]:
# Load CASP12 data from SidechainNet — keep in memory, no disk writes
import os
import torch
import numpy as np
import sidechainnet as scn

print('Loading SidechainNet CASP12 (thinning=30)...')
data = scn.load(casp_version=12, thinning=30)

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def seq_to_one_hot(seq_str):
    one_hot = torch.zeros(len(seq_str), 20)
    for i, aa in enumerate(seq_str):
        if aa in aa_to_idx:
            one_hot[i, aa_to_idx[aa]] = 1
    return one_hot

# Collect all protein data in memory: {(split, prot_id): {field: tensor}}
all_proteins = {}
for split_name in data.splits:
    prot_ids = data.split_to_ids.get(split_name, [])
    for prot_id in prot_ids:
        try:
            protein = data[prot_id]
        except (KeyError, Exception):
            continue
        seq = protein.seq
        if len(seq) < 10 or len(seq) > 600:
            continue
        coords = protein.coords
        if coords is None or len(coords) == 0:
            continue
        if isinstance(coords, np.ndarray):
            coords = torch.tensor(coords, dtype=torch.float32)
        backbone = coords[:, :3, :].clone()
        mask_str = protein.mask
        one_hot = seq_to_one_hot(seq)
        angles = protein.angles
        if angles is not None:
            ang = torch.tensor(np.array(angles), dtype=torch.float32) if not isinstance(angles, torch.Tensor) else angles.float()
        else:
            ang = torch.zeros(len(seq), 12)

        prot_id_clean = prot_id.replace('/', '_').replace('\\', '_')
        all_proteins[(split_name, prot_id_clean)] = {
            'id': prot_id,
            'seq': seq,
            'crd_backbone': backbone,
            'mask': mask_str,
            'seq_one_hot': one_hot,
            'ang': ang,
        }

# Free SidechainNet from memory
del data
print(f'Loaded {len(all_proteins)} proteins into memory')

@> ProDy is configured: verbosity='none'
INFO:.prody:ProDy is configured: verbosity='none'
INFO:.prody:ProDy is configured: auto_secondary=False


Loading SidechainNet CASP12 (thinning=30)...
SidechainNet(12, 30) was not found in ./sidechainnet_data.


Downloaded SidechainNet to ./sidechainnet_data/sidechainnet_casp12_30.pkl.
SidechainNet was loaded from ./sidechainnet_data/sidechainnet_casp12_30.pkl.
Loaded 24027 proteins into memory


In [ ]:
import re
import gc
import time
from transformers import T5EncoderModel, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model_name = 'Rostlab/prot_t5_xl_half_uniref50-enc'
model = T5EncoderModel.from_pretrained(model_name).half().to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('Model loaded.')

Device: cuda
GPU: Tesla T4


Exception: You're trying to run a `Unigram` model but you're file was trained with a different algorithm

In [ ]:
BATCH_SIZE = 32

def prep_seq(seq):
    return ' '.join(list(re.sub(r'[UZOB]', 'X', seq)))

def get_embeddings_batch(sequences):
    """Returns list of [seq_len, 1024] CPU float32 tensors. Cleans GPU after each call."""
    prepped = [prep_seq(s) for s in sequences]
    ids = tokenizer(prepped, add_special_tokens=True, padding='longest', return_tensors='pt')
    input_ids = ids['input_ids'].to(device)
    attention_mask = ids['attention_mask'].to(device)
    with torch.inference_mode():
        result = model(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = [result.last_hidden_state[i, :len(seq)].cpu().float() for i, seq in enumerate(sequences)]
    del result, input_ids, attention_mask, ids
    torch.cuda.empty_cache()
    return embeddings

def cycle_seq(seq, shift):
    return seq[shift:] + seq[:shift]

def make_mutant(seq):
    idx = torch.randperm(len(seq))[:2]
    s = list(seq)
    s[idx[0]], s[idx[1]] = s[idx[1]], s[idx[0]]
    return ''.join(s)

# cycle5 (shift=2) and cycle6 (shift=5) are unused in training
EMB_VARIANTS = [
    ('proT5_emb', None),
    ('proT5_emb_mut', 'mut'),
    ('proT5_emb_cycle1', -1),
    ('proT5_emb_cycle2', 1),
    ('proT5_emb_cycle3', -2),
    ('proT5_emb_cycle4', -5),
]

In [ ]:
# Build sorted work list: [(key, seq)] sorted by length for efficient batching
work_list = [(key, pdata['seq']) for key, pdata in all_proteins.items()]
work_list.sort(key=lambda x: len(x[1]))
print(f'{len(work_list)} proteins to process, sorted by length')

In [ ]:
# Generate all embeddings — checkpoint each variant to a single file on disk
import json

CHECKPOINT_DIR = '/content/emb_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

t0 = time.time()

for var_name, variant in EMB_VARIANTS:
    ckpt_path = os.path.join(CHECKPOINT_DIR, var_name + '.pt')

    # Load existing checkpoint if crashed mid-variant
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, weights_only=False)
        print(f'\n{var_name}: resuming from checkpoint ({len(ckpt)} already done)')
    else:
        ckpt = {}

    # Build todo list, skipping already-done proteins
    todo_keys = []
    todo_seqs = []
    for key, seq in work_list:
        str_key = f'{key[0]}/{key[1]}'
        if str_key in ckpt:
            continue
        if variant is None:
            s = seq
        elif variant == 'mut':
            s = make_mutant(seq)
        else:
            s = cycle_seq(seq, variant)
        todo_keys.append(key)
        todo_seqs.append(s)

    if not todo_keys:
        print(f'\n{var_name}: all done (checkpoint), skipping')
        # Store in memory from checkpoint
        for key, seq in work_list:
            str_key = f'{key[0]}/{key[1]}'
            if str_key in ckpt:
                all_proteins[key][var_name] = ckpt[str_key]
                if variant == 'mut' and str_key + '_seq' in ckpt:
                    all_proteins[key]['seq_mut'] = ckpt[str_key + '_seq']
        del ckpt
        continue

    print(f'\n{var_name}: {len(todo_keys)} sequences remaining')
    var_t0 = time.time()

    for start in range(0, len(todo_keys), BATCH_SIZE):
        batch_keys = todo_keys[start:start + BATCH_SIZE]
        batch_seqs = todo_seqs[start:start + BATCH_SIZE]

        try:
            embs = get_embeddings_batch(batch_seqs)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                gc.collect()
                half = max(1, len(batch_seqs) // 2)
                embs = get_embeddings_batch(batch_seqs[:half]) + get_embeddings_batch(batch_seqs[half:])
            else:
                raise

        for key, var_seq, emb in zip(batch_keys, batch_seqs, embs):
            str_key = f'{key[0]}/{key[1]}'
            ckpt[str_key] = emb
            all_proteins[key][var_name] = emb
            if variant == 'mut':
                ckpt[str_key + '_seq'] = var_seq
                all_proteins[key]['seq_mut'] = var_seq

        del embs

        done = min(start + BATCH_SIZE, len(todo_keys))

        # Checkpoint every 500 sequences
        if done % 500 < BATCH_SIZE or done == len(todo_keys):
            torch.save(ckpt, ckpt_path)
            elapsed = time.time() - var_t0
            rate = done / elapsed if elapsed > 0 else 0
            eta = (len(todo_keys) - done) / rate / 60 if rate > 0 else 0
            print(f'  {done}/{len(todo_keys)} ({elapsed:.0f}s, {rate:.1f} seq/s, ETA {eta:.0f}min) [saved]')

        if start % (BATCH_SIZE * 10) == 0:
            gc.collect()

    del ckpt, todo_keys, todo_seqs
    gc.collect()

elapsed = time.time() - t0
print(f'\nAll embeddings generated in {elapsed/60:.1f} minutes!')

In [ ]:
# Write everything to disk in one pass — per-protein directories matching expected format
from pathlib import Path

OUT_DIR = Path('/content/casp12_data_30')
emb_file_names = [name + '.pt' for name, _ in EMB_VARIANTS]

count = 0
for (split_name, prot_id), pdata in all_proteins.items():
    prot_dir = OUT_DIR / split_name / prot_id
    prot_dir.mkdir(parents=True, exist_ok=True)

    # Base fields
    torch.save(pdata['id'], str(prot_dir / 'id.pt'))
    torch.save(pdata['seq'], str(prot_dir / 'seq.pt'))
    torch.save(pdata['crd_backbone'], str(prot_dir / 'crd_backbone.pt'))
    torch.save(pdata['mask'], str(prot_dir / 'mask.pt'))
    torch.save(pdata['seq_one_hot'], str(prot_dir / 'seq_one_hot.pt'))
    torch.save(pdata['ang'], str(prot_dir / 'ang.pt'))

    # Embeddings
    for var_name, _ in EMB_VARIANTS:
        if var_name in pdata:
            torch.save(pdata[var_name], str(prot_dir / (var_name + '.pt')))
    if 'seq_mut' in pdata:
        torch.save(pdata['seq_mut'], str(prot_dir / 'seq_mut.pt'))

    count += 1
    if count % 2000 == 0:
        print(f'  Written {count}/{len(all_proteins)}...')

print(f'Done: {count} proteins written to {OUT_DIR}')

In [ ]:
# Tar and copy to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Free memory before tar
del all_proteins, work_list
gc.collect()

!tar -czf /content/casp12_data_30_with_embeddings.tar.gz -C /content casp12_data_30
print('Created tar.gz')

!cp /content/casp12_data_30_with_embeddings.tar.gz /content/drive/MyDrive/
print('Copied to Google Drive — download from there')